In [0]:
#Create a SparkSession.
from pyspark.sql.functions import *
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Customer Product Review Sentiment Analysis using PySpark NLP").getOrCreate()



In [0]:
#Read the table,display the schema and show the records:
reviews_df = spark.read.table('dbacademy.default.customer_reviews')
reviews_df.show(10)
reviews_df.printSchema()
reviews_df.describe().show()

+---------+-----------+-----+-----+--------+----------+----------+--------------------+------+----------+---------+
|ProductID|ProductName|Brand|Price|ReviewID|CustomerID|  Category|          ReviewText|Rating|ReviewDate|  Country|
+---------+-----------+-----+-----+--------+----------+----------+--------------------+------+----------+---------+
|     P501|       NULL| NULL| NULL|       1|      C101|    Mobile|Excellent camera ...|     5|2026-01-10|    India|
|     P502|       NULL| NULL| NULL|       2|      C102|    Laptop|Laptop hangs freq...|     2|2026-01-11|    India|
|     P503|       NULL| NULL| NULL|       3|      C103|Headphones|Sound quality is ...|     5|2026-01-12|      USA|
|     P504|       NULL| NULL| NULL|       4|      C104|    Mobile|Battery drains ve...|     2|2026-01-13|   Canada|
|     P501|       NULL| NULL| NULL|       5|      C105|    Mobile|Very good display...|     5|2026-01-14|    India|
|     P505|       NULL| NULL| NULL|       6|      C106|Television|Pictur

In [0]:
#Count total reviews.
total_reviews = reviews_df.count()

print(f"Total Reviews: {total_reviews}")

Total Reviews: 38


In [0]:
##Data Cleaning:
#Check for null values.
reviews_df.select(
    [
        count(when(col(column).isNull(), column)).alias(column)
        for column in reviews_df.columns
    ]
).show()

+---------+-----------+-----+-----+--------+----------+--------+----------+------+----------+-------+
|ProductID|ProductName|Brand|Price|ReviewID|CustomerID|Category|ReviewText|Rating|ReviewDate|Country|
+---------+-----------+-----+-----+--------+----------+--------+----------+------+----------+-------+
|        0|         20|   20|   20|      18|        18|      18|        18|    18|        18|     18|
+---------+-----------+-----+-----+--------+----------+--------+----------+------+----------+-------+



In [0]:
#Remove Duplicate Reviews
before_count = reviews_df.count()

reviews_df = reviews_df.dropDuplicates()

after_count = reviews_df.count()

print(f"Records Before Cleaning : {before_count}")
print(f"Records After Cleaning  : {after_count}")
print(f"Duplicates Removed      : {before_count - after_count}")

Records Before Cleaning : 38
Records After Cleaning  : 38
Duplicates Removed      : 0


In [0]:
#Convert ReviewDate into DateType.
reviews_df = reviews_df.withColumn(
    "ReviewDate",
    to_date(col("ReviewDate"), "yyyy-MM-dd")
)

In [0]:
#Trim spaces from ReviewText.
reviews_df = reviews_df.withColumn(
    "ReviewText",
    trim(col("ReviewText"))
)

In [0]:
#Convert Text to Lowercase
reviews_df = reviews_df.withColumn(
    "ReviewText",
    lower(col("ReviewText"))
)

In [0]:
# Filter out null ReviewText before tokenization
reviews_df = reviews_df.filter(col("ReviewText").isNotNull())

In [0]:
##NLP Processing
#Tokenize Review Text
from pyspark.ml.feature import Tokenizer

tokenizer = Tokenizer(
    inputCol="ReviewText",
    outputCol="Tokens"
)

reviews_df = tokenizer.transform(reviews_df)

In [0]:
#Remove stop words.
from pyspark.ml.feature import StopWordsRemover

stopword_remover = StopWordsRemover(
    inputCol="Tokens",
    outputCol="FilteredTokens"
)

reviews_df = stopword_remover.transform(reviews_df)

In [0]:
#Calculate total words in each review.
reviews_df = reviews_df.withColumn(
    "TotalWords",
    size(col("FilteredTokens"))
)

In [0]:
#Remove punctuation.
reviews_df = reviews_df.withColumn(
    "CleanReview",
    regexp_replace(col("ReviewText"), "[^a-zA-Z0-9\\s]", "")
)

In [0]:
#Remove numeric values from reviews.
reviews_df = reviews_df.withColumn(
    "CleanReview",
    regexp_replace(col("CleanReview"), "\\d+", "")
)

In [0]:
#Feature Engineering
#Create a column for review length.
reviews_df = reviews_df.withColumn(
    "ReviewLength",
    length(col("CleanReview"))
)


In [0]:
#Count unique words in each review.
reviews_df = reviews_df.withColumn(
    "UniqueWordCount",
    size(array_distinct(col("FilteredTokens")))
)

In [0]:
#Find average review length by category.
reviews_df.groupBy("Category").agg(avg("ReviewLength").alias("AverageReviewLength")) \
    .orderBy("Category") \
    .show(truncate=False)

+----------+-------------------+
|Category  |AverageReviewLength|
+----------+-------------------+
|Camera    |26.0               |
|Headphones|19.5               |
|Laptop    |28.333333333333332 |
|Mobile    |31.6               |
|Printer   |28.0               |
|Speaker   |22.5               |
|Tablet    |31.0               |
|Television|26.0               |
|Watch     |22.0               |
+----------+-------------------+



In [0]:
#Find the longest review.
reviews_df.orderBy(
    col("ReviewLength").desc()
).select(
    "ReviewID",
    "ProductName",
    "Category",
    "ReviewLength",
    "CleanReview"
).show(1, truncate=False)

+--------+-----------+--------+------------+-------------------------------------------+
|ReviewID|ProductName|Category|ReviewLength|CleanReview                                |
+--------+-----------+--------+------------+-------------------------------------------+
|1       |NULL       |Mobile  |43          |excellent camera quality and battery backup|
+--------+-----------+--------+------------+-------------------------------------------+
only showing top 1 row


In [0]:
#Find the shortest review.
reviews_df.orderBy(
    col("ReviewLength").asc()
).select(
    "ReviewID",
    "ProductName",
    "Category",
    "ReviewLength",
    "CleanReview"
).show(1, truncate=False)

+--------+-----------+----------+------------+---------------+
|ReviewID|ProductName|Category  |ReviewLength|CleanReview    |
+--------+-----------+----------+------------+---------------+
|9       |NULL       |Headphones|15          |bass is too low|
+--------+-----------+----------+------------+---------------+
only showing top 1 row


In [0]:
#Word Analysis
#Explode Tokens  to aggregate individual words inside an array of Filtered Tokens:
from pyspark.sql.functions import explode

words_df = reviews_df.select( "Category",explode(col("FilteredTokens")).alias("Word"))

In [0]:
#Find the top 20 most frequent words.
words_df.groupBy("Word") \
    .count() \
    .orderBy(col("count").desc()) \
    .show(20, truncate=False)

+------------+-----+
|Word        |count|
+------------+-----+
|quality     |4    |
|sound       |3    |
|display     |2    |
|performance.|2    |
|stopped     |2    |
|fast.       |2    |
|excellent   |2    |
|battery     |2    |
|phone       |1    |
|gets        |1    |
|heated      |1    |
|gaming.     |1    |
|good        |1    |
|smooth      |1    |
|bass        |1    |
|low.        |1    |
|laptop      |1    |
|hangs       |1    |
|frequently  |1    |
|update.     |1    |
+------------+-----+
only showing top 20 rows


In [0]:
#Find the top 10 words used in Mobile reviews.
words_df.filter(col("Category") == "Mobile") \
    .groupBy("Word") \
    .count() \
    .orderBy(col("count").desc()) \
    .show(10, truncate=False)

+------------+-----+
|Word        |count|
+------------+-----+
|battery     |2    |
|phone       |1    |
|gets        |1    |
|heated      |1    |
|gaming.     |1    |
|good        |1    |
|display     |1    |
|smooth      |1    |
|performance.|1    |
|charging    |1    |
+------------+-----+
only showing top 10 rows


In [0]:
#Find the top 10 words used in Laptop reviews.
words_df.filter(col("Category") == "Laptop") \
    .groupBy("Word") \
    .count() \
    .orderBy(col("count").desc()) \
    .show(10, truncate=False)



+-----------+-----+
|Word       |count|
+-----------+-----+
|laptop     |1    |
|hangs      |1    |
|frequently |1    |
|update.    |1    |
|keyboard   |1    |
|stopped    |1    |
|working.   |1    |
|lightweight|1    |
|fast.      |1    |
+-----------+-----+



In [0]:
#Count how many reviews contain the word battery.
reviews_df.filter(array_contains(col("FilteredTokens"), "battery")).count()

2

In [0]:
#Count how many reviews contain the word excellent.
reviews_df.filter(array_contains(col("FilteredTokens"), "excellent")).count()

2

In [0]:
##Rating Analysis
#Find average rating by category.
reviews_df.groupBy("Category") \
    .agg(avg("Rating").alias("AverageRating")) \
    .orderBy(col("AverageRating").desc()) \
    .show()

+----------+------------------+
|  Category|     AverageRating|
+----------+------------------+
|     Watch|               4.5|
|   Speaker|               4.0|
|   Printer|               4.0|
|    Camera|               3.5|
|Headphones|               3.5|
|    Mobile|               3.2|
|    Laptop|2.6666666666666665|
|Television|               2.0|
|    Tablet|               1.0|
+----------+------------------+



In [0]:
#Find average rating by country.
reviews_df.groupBy("Country").agg(avg("Rating").alias("AverageRating")) \
    .orderBy(col("AverageRating").desc()) \
    .show()



+---------+------------------+
|  Country|     AverageRating|
+---------+------------------+
|       UK|               3.5|
|    India|3.4444444444444446|
|      USA|              3.25|
|   Canada|2.6666666666666665|
|Australia|               2.5|
+---------+------------------+



In [0]:
#Count reviews for each rating.
reviews_df.groupBy("Rating") \
    .count() \
    .orderBy("Rating") \
    .show()

+------+-----+
|Rating|count|
+------+-----+
|     1|    3|
|     2|    6|
|     3|    2|
|     4|    2|
|     5|    7|
+------+-----+



In [0]:
#Find products having average rating below 3.
reviews_df.groupBy(
    "ProductID",
    "ProductName"
).agg(avg("Rating").alias("AverageRating")
).filter(
    col("AverageRating") < 3
).show(truncate=False)

+---------+-----------+-------------+
|ProductID|ProductName|AverageRating|
+---------+-----------+-------------+
|P502     |NULL       |2.0          |
|P504     |NULL       |2.0          |
|P506     |NULL       |1.0          |
|P509     |NULL       |2.0          |
|P511     |NULL       |1.0          |
|P513     |NULL       |2.0          |
|P515     |NULL       |1.0          |
|P518     |NULL       |2.0          |
+---------+-----------+-------------+



In [0]:
#Find products having average rating above 4.
reviews_df.groupBy(
    "ProductID",
    "ProductName"
).agg(
    avg("Rating").alias("AverageRating")
).filter( 
    col("AverageRating") > 4
).show(truncate=False)

+---------+-----------+-------------+
|ProductID|ProductName|AverageRating|
+---------+-----------+-------------+
|P501     |NULL       |5.0          |
|P507     |NULL       |5.0          |
|P508     |NULL       |5.0          |
|P510     |NULL       |5.0          |
|P517     |NULL       |5.0          |
+---------+-----------+-------------+



In [0]:
##Sentiment Preparation
#Create a sentiment column:
reviews_df = reviews_df.withColumn(
    "Sentiment",
    when(col("Rating") >= 4, "Positive")
    .when(col("Rating") == 3, "Neutral")
    .otherwise("Negative")
)


In [0]:
#Count positive reviews.
reviews_df.filter(
    col("Sentiment") == "Positive"
).count()

9

In [0]:
#Count neutral reviews.
reviews_df.filter(
    col("Sentiment") == "Neutral"
).count()



2

In [0]:
#Count negative reviews.
reviews_df.filter(
    col("Sentiment") == "Negative"
).count()




9

In [0]:
#Find sentiment distribution by category.
reviews_df.groupBy(
    "Category",
    "Sentiment"
).count() \
.orderBy(
    "Category",
    "Sentiment"
).show()

+----------+---------+-----+
|  Category|Sentiment|count|
+----------+---------+-----+
|    Camera| Negative|    1|
|    Camera| Positive|    1|
|Headphones| Negative|    1|
|Headphones| Positive|    1|
|    Laptop| Negative|    2|
|    Laptop| Positive|    1|
|    Mobile| Negative|    3|
|    Mobile| Positive|    2|
|   Printer| Positive|    1|
|   Speaker|  Neutral|    1|
|   Speaker| Positive|    1|
|    Tablet| Negative|    1|
|Television| Negative|    1|
|Television|  Neutral|    1|
|     Watch| Positive|    2|
+----------+---------+-----+



In [0]:
##NLP Features
#Generate TF (Term Frequency).
from pyspark.ml.feature import HashingTF

hashing_tf = HashingTF(
    inputCol="FilteredTokens",
    outputCol="TF",
    numFeatures=1000
)

tf_df = hashing_tf.transform(reviews_df)


In [0]:
#Generate IDF features.
from pyspark.ml.feature import IDF
idf = IDF(
    inputCol="TF",
    outputCol="TFIDF"
)

idf_model = idf.fit(tf_df)

In [0]:
#Create TF-IDF vectors.
tfidf_df = idf_model.transform(tf_df)

In [0]:
#Display TF-IDF vector size.
from pyspark.ml.functions import vector_to_array
tfidf_df.select(
    size(vector_to_array(col("TFIDF"))).alias("VectorSize")
).show(5)

+----------+
|VectorSize|
+----------+
|      1000|
|      1000|
|      1000|
|      1000|
|      1000|
+----------+
only showing top 5 rows


In [0]:
#Save transformed data.
tfidf_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dbacademy.default.customer_reviews_tfidf")

In [0]:
##Aggregation
#Find category with the highest average review length.
reviews_df.groupBy("Category").agg(avg("ReviewLength").alias("AverageReviewLength")) \
    .orderBy(col("AverageReviewLength").desc()) \
    .show(1, truncate=False)

+--------+-------------------+
|Category|AverageReviewLength|
+--------+-------------------+
|Mobile  |31.6               |
+--------+-------------------+
only showing top 1 row


In [0]:
#Find country with the highest number of reviews.
reviews_df.groupBy("Country") \
    .count() \
    .orderBy(col("count").desc()) \
    .show(1)

+-------+-----+
|Country|count|
+-------+-----+
|  India|    9|
+-------+-----+
only showing top 1 row


In [0]:
#Find top-rated category.
reviews_df.groupBy("Category").agg(avg("Rating").alias("AverageRating")) \
    .orderBy(col("AverageRating").desc()) \
    .show(1)

+--------+-------------+
|Category|AverageRating|
+--------+-------------+
|   Watch|          4.5|
+--------+-------------+
only showing top 1 row


In [0]:
#Find Lowest-Rated Category
reviews_df.groupBy("Category") \
    .agg(avg("Rating").alias("AverageRating")) \
    .orderBy(col("AverageRating")) \
    .show(1)

+--------+-------------+
|Category|AverageRating|
+--------+-------------+
|  Tablet|          1.0|
+--------+-------------+
only showing top 1 row


In [0]:
#Find products reviewed more than once.
reviews_df.groupBy(
    "ProductID",
    "ProductName"
).count() \
.filter(
    col("count") > 1
).show(truncate=False)

+---------+-----------+-----+
|ProductID|ProductName|count|
+---------+-----------+-----+
|P501     |NULL       |2    |
|P503     |NULL       |2    |
+---------+-----------+-----+



In [0]:
##Window Functions
#Rank products by average rating within each category.
from pyspark.sql.window import Window
product_rating = reviews_df.groupBy(
    "Category",
    "ProductID",
    "ProductName"
).agg(avg("Rating").alias("AverageRating"))

category_window_spec = Window.partitionBy("Category") \
    .orderBy(col("AverageRating").desc())

product_rating.withColumn(
    "Rank",
    dense_rank().over(category_window_spec)
).show(truncate=False)





+----------+---------+-----------+-------------+----+
|Category  |ProductID|ProductName|AverageRating|Rank|
+----------+---------+-----------+-------------+----+
|Camera    |P508     |NULL       |5.0          |1   |
|Camera    |P513     |NULL       |2.0          |2   |
|Headphones|P503     |NULL       |3.5          |1   |
|Laptop    |P517     |NULL       |5.0          |1   |
|Laptop    |P502     |NULL       |2.0          |2   |
|Laptop    |P506     |NULL       |1.0          |3   |
|Mobile    |P501     |NULL       |5.0          |1   |
|Mobile    |P504     |NULL       |2.0          |2   |
|Mobile    |P509     |NULL       |2.0          |2   |
|Mobile    |P518     |NULL       |2.0          |2   |
|Printer   |P512     |NULL       |4.0          |1   |
|Speaker   |P510     |NULL       |5.0          |1   |
|Speaker   |P516     |NULL       |3.0          |2   |
|Tablet    |P511     |NULL       |1.0          |1   |
|Television|P505     |NULL       |3.0          |1   |
|Television|P515     |NULL  

In [0]:
#Find the latest review for every product.
product_window_spec = Window.partitionBy("ProductID") \
    .orderBy(col("ReviewDate").desc())

reviews_df.withColumn(
    "RowNum",
    row_number().over(product_window_spec)
).filter(
    col("RowNum") == 1
).drop("RowNum") \
.show(truncate=False)

+---------+-----------+-----+-----+--------+----------+----------+-----------------------------------------+------+----------+---------+------------------------------------------------+--------------------------------------+----------+----------------------------------------+------------+---------------+---------+
|ProductID|ProductName|Brand|Price|ReviewID|CustomerID|Category  |ReviewText                               |Rating|ReviewDate|Country  |Tokens                                          |FilteredTokens                        |TotalWords|CleanReview                             |ReviewLength|UniqueWordCount|Sentiment|
+---------+-----------+-----+-----+--------+----------+----------+-----------------------------------------+------+----------+---------+------------------------------------------------+--------------------------------------+----------+----------------------------------------+------------+---------------+---------+
|P501     |NULL       |NULL |NULL |5       |C105    

In [0]:
#Find the earliest review for every customer.
customer_window_spec = Window.partitionBy("CustomerID") \
    .orderBy(col("ReviewDate"))

reviews_df.withColumn(
    "RowNum",
    row_number().over(customer_window_spec)
).filter(
    col("RowNum") == 1
).drop("RowNum") \
.show(truncate=False)

+---------+-----------+-----+-----+--------+----------+----------+--------------------------------------------+------+----------+---------+---------------------------------------------------+----------------------------------------------+----------+-------------------------------------------+------------+---------------+---------+
|ProductID|ProductName|Brand|Price|ReviewID|CustomerID|Category  |ReviewText                                  |Rating|ReviewDate|Country  |Tokens                                             |FilteredTokens                                |TotalWords|CleanReview                                |ReviewLength|UniqueWordCount|Sentiment|
+---------+-----------+-----+-----+--------+----------+----------+--------------------------------------------+------+----------+---------+---------------------------------------------------+----------------------------------------------+----------+-------------------------------------------+------------+---------------+---------+
|

In [0]:
#Find running total of reviews by date.
daily_reviews = reviews_df.groupBy("ReviewDate") \
    .count()

review_window_spec = Window.orderBy("ReviewDate") \
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)

daily_reviews.withColumn(
    "RunningTotalReviews",
    sum("count").over(review_window_spec)
).show(truncate=False)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+----------+-----+-------------------+
|ReviewDate|count|RunningTotalReviews|
+----------+-----+-------------------+
|2026-01-10|1    |1                  |
|2026-01-11|1    |2                  |
|2026-01-12|1    |3                  |
|2026-01-13|1    |4                  |
|2026-01-14|1    |5                  |
|2026-01-15|1    |6                  |
|2026-01-16|1    |7                  |
|2026-01-17|1    |8                  |
|2026-01-18|1    |9                  |
|2026-01-19|1    |10                 |
|2026-01-20|1    |11                 |
|2026-01-21|1    |12                 |
|2026-01-22|1    |13                 |
|2026-01-23|1    |14                 |
|2026-01-24|1    |15                 |
|2026-01-25|1    |16                 |
|2026-01-26|1    |17                 |
|2026-01-27|1    |18                 |
|2026-01-28|1    |19                 |
|2026-01-29|1    |20                 |
+----------+-----+-------------------+



In [0]:
#Assign row numbers within each category ordered by rating.
rating_window_spec = Window.partitionBy("Category") \
    .orderBy(col("Rating").desc())

reviews_df.withColumn(
    "RowNumber",
    row_number().over(rating_window_spec)
).show(truncate=False)

+---------+-----------+-----+-----+--------+----------+----------+--------------------------------------------+------+----------+---------+---------------------------------------------------+----------------------------------------------+----------+-------------------------------------------+------------+---------------+---------+---------+
|ProductID|ProductName|Brand|Price|ReviewID|CustomerID|Category  |ReviewText                                  |Rating|ReviewDate|Country  |Tokens                                             |FilteredTokens                                |TotalWords|CleanReview                                |ReviewLength|UniqueWordCount|Sentiment|RowNumber|
+---------+-----------+-----+-----+--------+----------+----------+--------------------------------------------+------+----------+---------+---------------------------------------------------+----------------------------------------------+----------+-------------------------------------------+------------+--------

In [0]:
##Join Tasks
#Join customer reviews with product master.
## Product master and customer review data were joined
# during the ingestion stage using ProductID.
# All subsequent analysis is performed on the merged DataFrame.



In [0]:
#Display ProductName with ReviewText.
reviews_df.select(
    "ProductName",
    "ReviewText"
).show(truncate=False)

+-----------+--------------------------------------------+
|ProductName|ReviewText                                  |
+-----------+--------------------------------------------+
|NULL       |excellent camera quality and battery backup.|
|NULL       |laptop hangs frequently after update.       |
|NULL       |sound quality is amazing.                   |
|NULL       |battery drains very fast.                   |
|NULL       |very good display and smooth performance.   |
|NULL       |picture quality is average.                 |
|NULL       |keyboard stopped working.                   |
|NULL       |looks premium and stylish.                  |
|NULL       |bass is too low.                            |
|NULL       |image quality is fantastic.                 |
|NULL       |charging is very slow.                      |
|NULL       |excellent sound clarity.                    |
|NULL       |display cracked within one week.            |
|NULL       |printing speed is impressive.              

In [0]:
#Average Rating by Brand
reviews_df.groupBy("Brand").agg(avg("Rating").alias("AverageRating")) \
    .orderBy(col("AverageRating").desc()) \
    .show()

+-----+-------------+
|Brand|AverageRating|
+-----+-------------+
| NULL|          3.2|
+-----+-------------+



In [0]:
#Total Products Reviewed for Each Brand
reviews_df.groupBy("Brand").agg(countDistinct("ProductID").alias("TotalProductsReviewed")) \
    .orderBy(col("TotalProductsReviewed").desc()) \
    .show()

+-----+---------------------+
|Brand|TotalProductsReviewed|
+-----+---------------------+
| NULL|                   18|
+-----+---------------------+



In [0]:
#Most Expensive Reviewed Product
reviews_df.orderBy(
    col("Price").desc()
).select(
    "ProductID",
    "ProductName",
    "Brand",
    "Price"
).show(1, truncate=False)

+---------+-----------+-----+-----+
|ProductID|ProductName|Brand|Price|
+---------+-----------+-----+-----+
|P501     |NULL       |NULL |NULL |
+---------+-----------+-----+-----+
only showing top 1 row


In [0]:
#Cheapest Reviewed Product
reviews_df.orderBy(
    col("Price")
).select(
    "ProductID",
    "ProductName",
    "Brand",
    "Price"
).show(1, truncate=False)

+---------+-----------+-----+-----+
|ProductID|ProductName|Brand|Price|
+---------+-----------+-----+-----+
|P501     |NULL       |NULL |NULL |
+---------+-----------+-----+-----+
only showing top 1 row


In [0]:
#Average Review Length by Brand
reviews_df.groupBy("Brand").agg(avg("ReviewLength").alias("AverageReviewLength")) \
    .orderBy(col("AverageReviewLength").desc()) \
    .show()

+-----+-------------------+
|Brand|AverageReviewLength|
+-----+-------------------+
| NULL|               26.7|
+-----+-------------------+



In [0]:
#Brand Having Highest Average Rating
reviews_df.groupBy("Brand").agg(avg("Rating").alias("AverageRating")) \
    .orderBy(col("AverageRating").desc()) \
    .show(1)

+-----+-------------+
|Brand|AverageRating|
+-----+-------------+
| NULL|          3.2|
+-----+-------------+



In [0]:
#Brand Having Lowest Average Rating
reviews_df.groupBy("Brand") .agg(avg("Rating").alias("AverageRating")) \
    .orderBy(col("AverageRating"))\
    .show(1)

+-----+-------------+
|Brand|AverageRating|
+-----+-------------+
| NULL|          3.2|
+-----+-------------+



In [0]:
spark.sql("SHOW VOLUMES IN workspace.default").show(truncate=False)
spark.sql("SHOW CATALOGS").show(truncate=False)
spark.sql("SHOW DATABASES").show(truncate=False)

+--------+-----------+
|database|volume_name|
+--------+-----------+
|default |my_files   |
+--------+-----------+

+---------+
|catalog  |
+---------+
|dbacademy|
|samples  |
|system   |
|workspace|
+---------+

+------------------+
|databaseName      |
+------------------+
|default           |
|information_schema|
+------------------+



In [0]:
#Save Final Output as PARQUET
reviews_df.write \
    .mode("overwrite") \
    .parquet("/Volumes/workspace/default/my_files/customer_review_analysis")

In [0]:
parquet_df = spark.read.parquet(
    "/Volumes/workspace/default/my_files/customer_review_analysis"
)

display(parquet_df)

ProductID,ProductName,Brand,Price,ReviewID,CustomerID,Category,ReviewText,Rating,ReviewDate,Country,Tokens,FilteredTokens,TotalWords,CleanReview,ReviewLength,UniqueWordCount,Sentiment
P518,null,null,null,20,C120,Mobile,phone gets heated while gaming.,2,2026-01-29,India,"List(phone, gets, heated, while, gaming.)","List(phone, gets, heated, gaming.)",4,phone gets heated while gaming,30,4,Negative
P501,null,null,null,5,C105,Mobile,very good display and smooth performance.,5,2026-01-14,India,"List(very, good, display, and, smooth, performance.)","List(good, display, smooth, performance.)",4,very good display and smooth performance,40,4,Positive
P503,null,null,null,9,C109,Headphones,bass is too low.,2,2026-01-18,Australia,"List(bass, is, too, low.)","List(bass, low.)",2,bass is too low,15,2,Negative
P502,null,null,null,2,C102,Laptop,laptop hangs frequently after update.,2,2026-01-11,India,"List(laptop, hangs, frequently, after, update.)","List(laptop, hangs, frequently, update.)",4,laptop hangs frequently after update,36,4,Negative
P511,null,null,null,13,C113,Tablet,display cracked within one week.,1,2026-01-22,India,"List(display, cracked, within, one, week.)","List(display, cracked, within, one, week.)",5,display cracked within one week,31,5,Negative
P513,null,null,null,15,C115,Camera,poor autofocus performance.,2,2026-01-24,USA,"List(poor, autofocus, performance.)","List(poor, autofocus, performance.)",3,poor autofocus performance,26,3,Negative
P514,null,null,null,16,C116,Watch,comfortable to wear.,4,2026-01-25,India,"List(comfortable, to, wear.)","List(comfortable, wear.)",2,comfortable to wear,19,2,Positive
P508,null,null,null,10,C110,Camera,image quality is fantastic.,5,2026-01-19,Canada,"List(image, quality, is, fantastic.)","List(image, quality, fantastic.)",3,image quality is fantastic,26,3,Positive
P515,null,null,null,17,C117,Television,remote stopped functioning.,1,2026-01-26,Canada,"List(remote, stopped, functioning.)","List(remote, stopped, functioning.)",3,remote stopped functioning,26,3,Negative
P506,null,null,null,7,C107,Laptop,keyboard stopped working.,1,2026-01-16,USA,"List(keyboard, stopped, working.)","List(keyboard, stopped, working.)",3,keyboard stopped working,24,3,Negative
